# Experiment 6 - RNN, LSTM and GRU for sequence learning and video understanding

Source of truth is `experiment_6.py`; this notebook mirrors it.

In [ ]:
"""Experiment 6: RNN, LSTM and GRU for sequence learning and video understanding."""
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import json
import time
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split

ROOT = Path(__file__).resolve().parent
DATA = ROOT / "data"
HAR_DIR = DATA / "UCI HAR Dataset"
VIDEO_DIR = DATA / "ucf101" / "UCF101_subset"
SEED = 42
SEQ_LEN = 128
N_CHANNELS = 9
N_ACTIVITIES = 6
UNITS = 32
DROPOUT = 0.2
BATCH = 32
EPOCHS = 30
PER_CLASS = 500
VIDEO_CLASSES = ["Archery", "BabyCrawling", "BalanceBeam", "BandMarching", "Basketball"]
VIDEO_FRAMES = 10
VIDEO_EPOCHS = 30
SEQ_LENGTHS = [32, 64, 128]
QUICK = os.environ.get("EXP6_QUICK") == "1"
if QUICK:
    PER_CLASS, EPOCHS, VIDEO_EPOCHS = 60, 2, 2
    SEQ_LENGTHS = [32, 128]

ACTIVITIES = ["WALKING", "WALKING_UPSTAIRS", "WALKING_DOWNSTAIRS", "SITTING", "STANDING", "LAYING"]

np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.keras.utils.set_random_seed(SEED)

CHANNELS = [
    "body_acc_x", "body_acc_y", "body_acc_z",
    "body_gyro_x", "body_gyro_y", "body_gyro_z",
    "total_acc_x", "total_acc_y", "total_acc_z",
]


def load_har_split(split):
    signal_dir = HAR_DIR / split / "Inertial Signals"
    channels = []
    for name in CHANNELS:
        channels.append(np.loadtxt(signal_dir / f"{name}_{split}.txt"))
    x = np.stack(channels, axis=-1).astype("float32")
    y = np.loadtxt(HAR_DIR / split / f"y_{split}.txt").astype("int64") - 1
    return x, y


def balanced_indices(y, per_class, rng):
    chosen = []
    for cls in range(N_ACTIVITIES):
        cls_idx = np.where(y == cls)[0]
        chosen.append(rng.choice(cls_idx, size=min(per_class, len(cls_idx)), replace=False))
    idx = np.concatenate(chosen)
    rng.shuffle(idx)
    return idx


def build_recurrent(kind, n_classes, seq_len=SEQ_LEN, units=UNITS):
    layer = {"rnn": layers.SimpleRNN, "lstm": layers.LSTM, "gru": layers.GRU}[kind]
    model = models.Sequential([
        layers.Input(shape=(seq_len, N_CHANNELS)),
        layer(units),
        layers.Dropout(DROPOUT),
        layers.Dense(16, activation="relu"),
        layers.Dense(n_classes, activation="softmax"),
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model


def train_model(model, x_tr, y_tr, x_val, y_val, epochs=EPOCHS):
    start = time.perf_counter()
    hist = model.fit(x_tr, y_tr, validation_data=(x_val, y_val),
                     epochs=epochs, batch_size=BATCH, verbose=0)
    return hist, time.perf_counter() - start


def score_model(model, x_test, y_test):
    probs = model.predict(x_test, batch_size=128, verbose=0)
    preds = np.argmax(probs, axis=1)
    return {
        "accuracy": float(accuracy_score(y_test, preds)),
        "precision": float(precision_score(y_test, preds, average="macro", zero_division=0)),
        "recall": float(recall_score(y_test, preds, average="macro", zero_division=0)),
        "f1": float(f1_score(y_test, preds, average="macro", zero_division=0)),
        "params": int(model.count_params()),
        "confusion": confusion_matrix(y_test, preds).tolist(),
        "predictions": preds.tolist(),
    }


def save_fig(fig, name):
    fig.tight_layout()
    fig.savefig(ROOT / name, dpi=130, bbox_inches="tight")
    plt.close(fig)


def bptt_exercise():
    x = [0.5, 0.7, 0.2]
    h, wx, wh, b = 0.0, 0.5, 0.8, 0.1
    values = []
    for xt in x:
        h = np.tanh(wx * xt + wh * h + b)
        values.append(float(h))
    return values


def load_video_frames(path, n_frames):
    import imageio.v2 as imageio
    reader = imageio.get_reader(str(path))
    total = reader.count_frames()
    idx = np.linspace(0, total - 1, n_frames).astype(int)
    frames = [tf.image.resize(reader.get_data(int(i)), (224, 224)).numpy().astype("uint8") for i in idx]
    reader.close()
    return np.stack(frames)

In [ ]:
results = {"seed": SEED, "per_class": PER_CLASS, "epochs": EPOCHS, "units": UNITS}

## HAR data

Load the raw inertial signals, build the balanced subset, normalise and split.

In [ ]:
# ---------------- HAR data ----------------
x_train_full, y_train_full = load_har_split("train")
x_test_full, y_test_full = load_har_split("test")
rng = np.random.default_rng(SEED)
subset_idx = balanced_indices(y_train_full, PER_CLASS, rng)
x_sub = x_train_full[subset_idx]
y_sub = y_train_full[subset_idx]

mean = x_sub.mean(axis=(0, 1), keepdims=True)
std = x_sub.std(axis=(0, 1), keepdims=True) + 1e-6
x_sub = (x_sub - mean) / std
x_test = (x_test_full - mean) / std

x_tr, x_val, y_tr, y_val = train_test_split(x_sub, y_sub, test_size=0.15, random_state=SEED, stratify=y_sub)
print(f"HAR train {x_tr.shape}  val {x_val.shape}  test {x_test.shape}")
results["har_shapes"] = {"train": list(x_tr.shape), "val": list(x_val.shape), "test": list(x_test.shape)}

## Temporal signals

Sensor channels over time for three activities and the class distribution.

In [ ]:
# ---------------- Plot 1: sensor signals ----------------
fig, axes = plt.subplots(3, 3, figsize=(14, 9))
show_classes = [0, 2, 3]
channel_ids = [0, 1, 6]
for row, (cls, ch) in enumerate(zip(show_classes, channel_ids)):
    seq = x_test[np.where(y_test_full == cls)[0][0]]
    for col in range(3):
        ax = axes[row, col]
        ax.plot(seq[:, col] if col < 3 else seq[:, ch], color="steelblue")
        ax.set_title(f"{ACTIVITIES[cls]}\n{CHANNELS[col]}", fontsize=8)
        ax.set_xlabel("time step")
        ax.grid(alpha=0.3)
fig.suptitle("Plot 1: sensor signals over time for three activities")
save_fig(fig, "exp6_plot_01_sensor_signals.png")

dist = np.bincount(y_sub, minlength=N_ACTIVITIES)
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(N_ACTIVITIES), dist, color="steelblue", edgecolor="black")
ax.set_xticks(range(N_ACTIVITIES))
ax.set_xticklabels([a.replace("WALKING_", "W_") for a in ACTIVITIES], rotation=30, ha="right", fontsize=8)
ax.set_ylabel("count")
ax.set_title("Class distribution of the training subset")
save_fig(fig, "exp6_plot_02_class_distribution.png")

## Backpropagation through time

Manual hidden-state calculation for the numerical exercise.

In [ ]:
# ---------------- BPTT numerical exercise ----------------
results["bptt"] = bptt_exercise()
print("BPTT h values:", results["bptt"])

## RNN, LSTM and GRU

Same input, optimiser, batch size and epochs for all three.

In [ ]:
# ---------------- RNN / LSTM / GRU ----------------
models_dict = {}
histories = {}
metrics = {}
for kind in ["rnn", "lstm", "gru"]:
    print(f"Training {kind.upper()} ...")
    model = build_recurrent(kind, N_ACTIVITIES)
    hist, elapsed = train_model(model, x_tr, y_tr, x_val, y_val)
    res = score_model(model, x_test, y_test_full)
    res["time_s"] = elapsed
    models_dict[kind] = model
    histories[kind] = hist.history
    metrics[kind] = res
    print(f"  {kind.upper()} acc {res['accuracy']:.4f}  f1 {res['f1']:.4f}  params {res['params']}  time {elapsed:.1f}s")
results["har_models"] = {k: {kk: vv for kk, vv in v.items() if kk != "confusion"} for k, v in metrics.items()}
results["har_confusion"] = {k: v["confusion"] for k, v in metrics.items()}

# Plots 2 and 3
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, kind in zip(axes, ["rnn", "lstm", "gru"]):
    h = histories[kind]
    ax.plot(h["loss"], label="train", color="steelblue")
    ax.plot(h["val_loss"], label="validation", color="darkorange")
    ax.set_title(f"{kind.upper()} loss")
    ax.set_xlabel("epoch"); ax.set_ylabel("loss"); ax.legend(); ax.grid(alpha=0.3)
fig.suptitle("Plot 2: training and validation loss")
save_fig(fig, "exp6_plot_03_loss.png")

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, kind in zip(axes, ["rnn", "lstm", "gru"]):
    h = histories[kind]
    ax.plot(np.array(h["accuracy"]) * 100, label="train", color="steelblue")
    ax.plot(np.array(h["val_accuracy"]) * 100, label="validation", color="darkorange")
    ax.set_title(f"{kind.upper()} accuracy")
    ax.set_xlabel("epoch"); ax.set_ylabel("accuracy (%)"); ax.legend(); ax.grid(alpha=0.3)
fig.suptitle("Plot 3: training and validation accuracy")
save_fig(fig, "exp6_plot_04_accuracy.png")

# Plot 4: confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
for ax, kind in zip(axes, ["rnn", "lstm", "gru"]):
    cm = np.array(metrics[kind]["confusion"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=[a[:6] for a in ACTIVITIES], yticklabels=[a[:6] for a in ACTIVITIES])
    ax.set_title(f"{kind.upper()}")
    ax.set_xlabel("predicted"); ax.set_ylabel("true")
fig.suptitle("Plot 4: confusion matrices on the test set")
save_fig(fig, "exp6_plot_05_confusion.png")

# Plot 5: comparison bars
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
kinds = ["rnn", "lstm", "gru"]
labels = [k.upper() for k in kinds]
accs = [metrics[k]["accuracy"] * 100 for k in kinds]
f1s = [metrics[k]["f1"] * 100 for k in kinds]
times = [metrics[k]["time_s"] for k in kinds]
axes[0].bar(labels, accs, color="steelblue", edgecolor="black"); axes[0].set_title("Test accuracy (%)")
axes[1].bar(labels, f1s, color="seagreen", edgecolor="black"); axes[1].set_title("Macro F1 (%)")
axes[2].bar(labels, times, color="darkorange", edgecolor="black"); axes[2].set_title("Training time (s)")
for ax in axes:
    ax.grid(axis="y", alpha=0.3)
    for bar in ax.patches:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f"{bar.get_height():.1f}",
                ha="center", va="bottom", fontsize=8)
fig.suptitle("Plot 5: RNN vs LSTM vs GRU")
save_fig(fig, "exp6_plot_06_model_comparison.png")

## Sequence length

Repeat the models with truncated sequences.

In [ ]:
# ---------------- sequence length study ----------------
print("Sequence length study ...")
seq_results = {}
for length in SEQ_LENGTHS:
    x_tr_t = x_tr[:, :length, :]
    x_val_t = x_val[:, :length, :]
    x_test_t = x_test[:, :length, :]
    seq_results[str(length)] = {}
    for kind in kinds:
        model = build_recurrent(kind, N_ACTIVITIES, seq_len=length)
        model.fit(x_tr_t, y_tr, validation_data=(x_val_t, y_val), epochs=EPOCHS, batch_size=BATCH, verbose=0)
        res = score_model(model, x_test_t, y_test_full)
        seq_results[str(length)][kind] = {"f1": res["f1"], "accuracy": res["accuracy"]}
        print(f"  T={length} {kind.upper()} f1 {res['f1']:.4f}")
results["sequence_length"] = seq_results

fig, ax = plt.subplots(figsize=(8, 4.5))
for kind in kinds:
    ax.plot(SEQ_LENGTHS, [seq_results[str(t)][kind]["f1"] * 100 for t in SEQ_LENGTHS], "o-", label=kind.upper())
ax.set_xlabel("sequence length")
ax.set_ylabel("test macro F1 (%)")
ax.set_title("Plot 6: sequence length vs test F1")
ax.legend(); ax.grid(alpha=0.3)
save_fig(fig, "exp6_plot_07_sequence_length.png")

## Video understanding

MobileNetV2 frame features followed by an LSTM and a GRU.

In [ ]:
# ---------------- video understanding ----------------
print("Video feature extraction ...")
base = tf.keras.applications.MobileNetV2(include_top=False, weights="imagenet", input_shape=(224, 224, 3))
extractor = models.Model(base.input, layers.GlobalAveragePooling2D()(base.output))
feat_dim = int(extractor.output_shape[-1])

videos, video_labels = [], []
for cls_id, cls in enumerate(VIDEO_CLASSES):
    for split in ["train", "val", "test"]:
        folder = VIDEO_DIR / split / cls
        if not folder.exists():
            continue
        for path in sorted(folder.glob("*.avi")):
            try:
                frames = load_video_frames(path, VIDEO_FRAMES)
            except Exception as exc:
                print(f"  skip {path.name}: {exc}")
                continue
            videos.append((cls_id, split, frames))
print(f"  loaded {len(videos)} videos, feature dim {feat_dim}")
results["video"] = {"feature_dim": feat_dim, "frames": VIDEO_FRAMES, "classes": VIDEO_CLASSES, "n_videos": len(videos)}

def extract_features(frames):
    frames = tf.keras.applications.mobilenet_v2.preprocess_input(frames.astype("float32"))
    return extractor.predict(frames, batch_size=32, verbose=0)

feats = {}
for cls_id, split, frames in videos:
    feats.setdefault(split, []).append((cls_id, extract_features(frames)))

def stack(split):
    xs = np.stack([f for _, f in feats[split]]).astype("float32")
    ys = np.array([c for c, _ in feats[split]])
    return xs, ys

xv_tr, yv_tr = stack("train")
xv_val, yv_val = stack("val")
xv_te, yv_te = stack("test")
n_video_classes = len(VIDEO_CLASSES)
print(f"  video train {xv_tr.shape} val {xv_val.shape} test {xv_te.shape}")

video_histories = {}
video_metrics = {}
for kind in ["lstm", "gru"]:
    print(f"Training video {kind.upper()} ...")
    layer = {"lstm": layers.LSTM, "gru": layers.GRU}[kind]
    model = models.Sequential([
        layers.Input(shape=(VIDEO_FRAMES, feat_dim)),
        layer(UNITS),
        layers.Dropout(DROPOUT),
        layers.Dense(16, activation="relu"),
        layers.Dense(n_video_classes, activation="softmax"),
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    start = time.perf_counter()
    hist = model.fit(xv_tr, yv_tr, validation_data=(xv_val, yv_val), epochs=VIDEO_EPOCHS, batch_size=8, verbose=0)
    elapsed = time.perf_counter() - start
    res = score_model(model, xv_te, yv_te)
    res["time_s"] = elapsed
    video_histories[kind] = hist.history
    video_metrics[kind] = res
    print(f"  video {kind.upper()} acc {res['accuracy']:.4f} f1 {res['f1']:.4f}")
results["video_models"] = {k: {kk: vv for kk, vv in v.items() if kk not in ("confusion", "predictions")} for k, v in video_metrics.items()}
results["video_confusion"] = {k: v["confusion"] for k, v in video_metrics.items()}

# Plot 7: sample frames
a_class, a_split, a_frames = videos[0]
fig, axes = plt.subplots(1, VIDEO_FRAMES, figsize=(18, 2.2))
for ax, frame in zip(axes, a_frames):
    ax.imshow(frame); ax.axis("off")
fig.suptitle(f"Plot 7: ten sampled frames from one {VIDEO_CLASSES[a_class]} video")
save_fig(fig, "exp6_plot_08_video_frames.png")

# Plot 8: video curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for kind in ["lstm", "gru"]:
    h = video_histories[kind]
    axes[0].plot(np.array(h["accuracy"]) * 100, label=f"{kind.upper()} train")
    axes[0].plot(np.array(h["val_accuracy"]) * 100, ls="--", label=f"{kind.upper()} val")
    axes[1].plot(h["loss"], label=f"{kind.upper()} train")
    axes[1].plot(h["val_loss"], ls="--", label=f"{kind.upper()} val")
axes[0].set_title("Accuracy (%)"); axes[1].set_title("Loss")
for ax in axes:
    ax.set_xlabel("epoch"); ax.legend(fontsize=7); ax.grid(alpha=0.3)
fig.suptitle("Plot 8: video training and validation curves")
save_fig(fig, "exp6_plot_09_video_curves.png")

# Plot 9: video confusion
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, kind in zip(axes, ["lstm", "gru"]):
    cm = np.array(video_metrics[kind]["confusion"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Greens", cbar=False, ax=ax,
                xticklabels=[c[:6] for c in VIDEO_CLASSES], yticklabels=[c[:6] for c in VIDEO_CLASSES])
    ax.set_title(kind.upper()); ax.set_xlabel("predicted"); ax.set_ylabel("true")
fig.suptitle("Plot 9: video confusion matrices")
save_fig(fig, "exp6_plot_10_video_confusion.png")

## Sequence to sequence

Encoder-decoder LSTM on a synthetic reversal task.

In [ ]:
# ---------------- sequence to sequence ----------------
print("Sequence-to-sequence reversal ...")
vocab = 10
seq_in = 8
n_samples = 4000 if not QUICK else 400
rng_seq = np.random.default_rng(SEED)
data = rng_seq.integers(1, vocab, size=(n_samples, seq_in)).astype("int64")
target = data[:, ::-1]
x_onehot = tf.keras.utils.to_categorical(data, vocab)
y_onehot = tf.keras.utils.to_categorical(target, vocab)

enc_in = layers.Input(shape=(seq_in, vocab))
enc = layers.LSTM(64)(enc_in)
rep = layers.RepeatVector(seq_in)(enc)
dec = layers.LSTM(64, return_sequences=True)(rep)
out = layers.TimeDistributed(layers.Dense(vocab, activation="softmax"))(dec)
seq2seq = models.Model(enc_in, out)
seq2seq.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="categorical_crossentropy", metrics=["accuracy"])
s2s_start = time.perf_counter()
s2s_hist = seq2seq.fit(x_onehot, y_onehot, validation_split=0.1, epochs=30, batch_size=32, verbose=0)
s2s_time = time.perf_counter() - s2s_start
preds = np.argmax(seq2seq.predict(x_onehot[:500], verbose=0), axis=-1)
token_acc = float((preds == target[:500]).mean())
seq_acc = float((preds == target[:500]).all(axis=1).mean())
print(f"  token acc {token_acc:.4f}  sequence acc {seq_acc:.4f}")
results["seq2seq"] = {
    "token_accuracy": token_acc, "sequence_accuracy": seq_acc, "params": int(seq2seq.count_params()),
    "train_time_s": s2s_time, "loss": s2s_hist.history["loss"], "val_loss": s2s_hist.history["val_loss"],
    "accuracy": s2s_hist.history["accuracy"], "val_accuracy": s2s_hist.history["val_accuracy"],
    "examples": [{"input": data[i].tolist(), "target": target[i].tolist(), "predicted": preds[i].tolist()} for i in range(5)],
}

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(s2s_hist.history["loss"], label="train"); axes[0].plot(s2s_hist.history["val_loss"], label="validation")
axes[0].set_title("seq2seq loss"); axes[0].set_xlabel("epoch"); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(np.array(s2s_hist.history["accuracy"]) * 100, label="train")
axes[1].plot(np.array(s2s_hist.history["val_accuracy"]) * 100, label="validation")
axes[1].set_title("token accuracy (%)"); axes[1].set_xlabel("epoch"); axes[1].legend(); axes[1].grid(alpha=0.3)
fig.suptitle("Plot 10: sequence-to-sequence training")
save_fig(fig, "exp6_plot_11_seq2seq.png")

## Consolidated results

Write the results JSON and the metrics summary.

In [ ]:
# ---------------- consolidated ----------------
lines = []
lines.append("Experiment 6 results (seed 42)")
lines.append("=" * 60)
lines.append("HAR sequence classification (test set):")
lines.append(f"{'Model':<8}{'Acc':>8}{'Prec':>8}{'Rec':>8}{'F1':>8}{'Params':>10}{'Time(s)':>10}")
for kind in kinds:
    m = metrics[kind]
    lines.append(f"{kind.upper():<8}{m['accuracy']:>8.4f}{m['precision']:>8.4f}{m['recall']:>8.4f}{m['f1']:>8.4f}{m['params']:>10}{m['time_s']:>10.1f}")
lines.append("")
lines.append("Sequence length vs test macro F1:")
for t in SEQ_LENGTHS:
    row = seq_results[str(t)]
    lines.append(f"  T={t}: " + "  ".join(f"{k.upper()} {row[k]['f1']:.4f}" for k in kinds))
lines.append("")
lines.append("Video (CNN features + recurrent):")
for kind in ["lstm", "gru"]:
    m = video_metrics[kind]
    lines.append(f"  {kind.upper()}: acc {m['accuracy']:.4f} f1 {m['f1']:.4f} params {m['params']} time {m['time_s']:.1f}s")
lines.append("")
lines.append(f"Seq2seq reversal: token acc {token_acc:.4f}  sequence acc {seq_acc:.4f}  params {seq2seq.count_params()}")
(ROOT / "experiment_6_metrics.txt").write_text("\n".join(lines) + "\n")
(ROOT / "experiment_6_results.json").write_text(json.dumps(results, indent=2))
print("Done. Results and figures written.")